# Глава 15. Фреймворки

Для начала рассмотрим инструменты для локального запуска моделей.

## Дескптопные решения

#### llama.cpp
[[llama.cpp]](https://github.com/ggml-org/llama.cpp) - появившийся в 2023 году легковесный фреймворк и утилита для инференса локальных языковых моделей на десктопном железе, альтернатива тяжелым нейросетевым SDK типа PyTorch. Детище болгарского разработчика Георгия Герганова, написанное на C++, которое получило признание в сообществе инженеров и для которого позже появились UI обертки, самые изветсные LM Studio и Ollama. На момент 2023 года, если разработчик хотел использовать возможности языковых моделей, выбор у него стоял между подключением платного API для облачных моделей типа GPT-4, и покупкой дорогой видеокарты на 24ГБ для скромных 7B мод. Целью Герганова был запуск только что вышедшей модели LLAMA-1 на своём макбуке. У него был написанный с нуля свой движок тензорных вычислений GGML, в котором он убрал все ненужные зависимости, оставив только мимнимальный набор команд для инференса.

Библиотека работает с жестко заданным списком доступных архитекутур (LLAMA, Qwen, Gemma и других), веса моделей должны быть скачены локально, в формате GGUF. В отличие от продакшен систем, ориентированных на параллельное вычисление CUDA ядрами, llama.cpp не требует загрузки целиком в память и, что главное, умеет работать с весами произвольной битности. Кроме того, это не серверное решение, оно не опитмизировано под поток запросов и не имеет хитрых механизмов батчинга, распредленного внимания и т.п., зато очень хорошо экономит память.

Можно использовать как утилиту, как API (есть команда `llamaserver`), как SDK: вызывать из кода на С++ или Python проектов. Современем добавили феукционал типа скачивания моделей из хаба HugginfaceF и сервинга HTTP зарпосов, но основая задачей остается инференс на потребительском железе. Из коробки бибилотека поддерживает спекулятивное декодирование, взяли три базовых режима: с внешней моделью, с внутренней моделью и с промптом. Из коробки RAG не поддерживается, но можно собрать - первый сервер запускается в режиме эмбедингов (`llamaserver --embeddings`), второй - модель для генерации ответа

#### GGML
Фундамент llama.cpp - это фреймворк тензорных вычислений GGML. Греганов написал его годом ранее в 2022. Написал его для модели Whisper - популярной модели рапознавания речи от OpenAI. вдозновившись минимализмом утилит, FFmpeg. Исходный код 1MB.

#### LM Studio
Спустя пару месяцев на основе библиотеки llama.cpp компания ElementLabs создала удобный пользовательский интерфейс, 

#### Ollama
В июле инженеры (ex-Docker) выпустили обёртку для llama.cpp, которая добавила удобное управление моделями в стиле Docker (), собрали свой репозиторий моделей, а также сделали более умный механизм использования памяти видеокарты. Ollama поддерживает сразу два API: свой нативный и стандартный OpenAI совместимый, можно отправлять запросы и туда, и туда.

Команды API соотвествуют стандарту ChatGPT и повторяют SDK-шные
- api/generate
- api/chat
- api/embed
- api/show
- api/create
- api/pull
- api/push
- api/delete
- api/copy
- api/tags
- api/ps

#### LocalAI
Логическое продолжение llama.cpp, вышедшее в 2023 году. Более гибкий вариант, собранный по принципу швейцарского ножа, скорее оркестратор. Поддерживает более широкий список модальностей (картинки, аудио), бекендов (llama,cpp, whisper, stablediffusion) и форматов (GGUF, OONX, Safetensors). Само приложение написано на Go, оно принимает запросы по REST API и маршрутизирует в нужный бекенд. Таким образом обеспечивается единая точка входа, когда моделей в окружении много и модели разные. Настройка YAML-файл 

#### AnythingLLM
Высокоуровневое решение от Mintplex Labs, вышедшее в 2024 году — конфигурируемый no-code конструктор для создания локальных LLM-приложений. Его серверная часть реализована в виде Node.js приложения, по REST, а фронтенд - это написанный на React чат-виджет, в браузере или интегрируемый с корпоративными базами знаний типа Confluence и Notion. Для ключевых действий приложение использует компоненты библиотеки LangChain. 

Система работает с любыми локальными и облачными моделями, соотвествующими стандарту OpenAI API. Наиболее полезная фича - работающий из коробки RAG-пайплайн: есть встроенные парсеры для PDF, DOCX и прочего, настраивается разбиение на чанки, есть векторизция - нативная модель или подключение к локальной / облачной, есть ранжирующая модель, а также векторное хранение - своя база или внешние базы типа QDrant. Также платформа поддерживает режим агента, то есть подключение инструментов, описание которых передается либо через аргумент запроса, либо прямо в промпт, а соотвествующий JS код кладется в локальный каталог. Второй важный компонент - это профили пользователя (workspaces), в каждом свои документы, системным промпты, модели и история. С помощью них реализуется разграничение прав.

#### SGLang

## Серверные решения

#### vLLM
Движок инференса промышленного уровня (production-ready), разработанный исследователями из Беркли. Если движки семейства llama.cpp создавались для экономии памяти на домашних ПК, то главная цель vLLM — максимизация пропускной способности (throughput) и скорости генерации на мощных серверных GPU (NVIDIA A100, H100, RTX 3090/4090). Фреймворк ориентирован на сценарии, когда к одной модели одновременно обращаются десятки, сотни или тысячи пользователей (веб-сервисы, корпоративные чат-боты, API-облака).

Техническая новация в том, что vLLM использует __PagedAttention__ - инженерную адаптацию алгоритма FlashAttetnion. Напомним, что FlashAttention - блочное вычисление весов внимания, где перемножение делается в быстрой видеопамяти SRAM. Но проблема в том, что сама нарезка на куски делалась вне перемножения. Традиционно память под контекст диалога (KV-кэш) выделялась одним большим куском. Написали ядра чтобы нарзека делалась до. 

Обрабатывать входящие запросы гигантскими динамическими пакетами (Continuous Batching). 
Фреймворк жестко завязан на экосистему CUDA и Triton, требует полной загрузки модели в видеопамять и не оптимизирован под работу на обычных CPU, что делает его де-факто стандартом для серверного бэкенда, но избыточным решением для индивидуального домашнего использования.

## Облачные решения
ыва



## Открытый стандарт языкового API
Для моделей машинного обучения REST API - наиболее удобный способ коммуникации, особенно если это продакшен решений.  Как правило, в начале разивтия технологии появиляется зоопарк конкурирующих интерфейсов, но как только рынок стабилизируется все переходят на универсальный протокол. Необходимость учить новый формат - фактор усложняющий адаптацию решения.

Напомним, что .  Stateless и .

В 2020 году OpenAI выпустили свою первую действительно большую модель GPT-3 на 175 млрд параметров [(Brown et al., 2020)](Brown et al., 2020). Чтобы окупить дорогостоящее обучение (по оценкам более 10 млн долларов), они открыли платный доступ по API, по сути создав новую бизнес модель - предобученная языковая модель по подписке. Разработчики смогли использовать возможности в своих приложениях без необходимости покупки дорогостоящего железа.

В конце 2022 года у OpenAI вышла знаковая модель ChatGPT, доступная в формате веб-интерфейса и работающая режиме чата. Она "взорвала" рынок, получив милиион пользователй за 5 дней и сделала OpenAI безогворочными лидером индустрии. Содержание в messages, и разделили на роли (user, system, assistant), а имя вызываемой модели вынесено в параметр "model". Веб приложение собрало миллион польователей за 5 дней. Конкуренты стали делать в таком же формате, чтобы было проще мигрировать.
В марте 2023 года вышла мультимодальная GPT-4, где появилась возможность прикреплять изображения в параметре "images". А через пару месяцев подключать инструменты в параметре tools. 

С течением времени конкуренция выросла и стало появляться все больше открытых моделей. Чтобы обеспечить взаимозаменяемость и обратную совместимость, проихводители стали реализоваывть свой API по образу ставшего стандаратом API от OpenAI.

Что именно стандартизируется? Во-первых, названия ключевых эндпойнтов: __v1/model__ для получения списка доступных моделей __v1/chat/completions__ для генерации продолжения (режим GPT) и __v1/embeddings__ для получения эмбедингов промпта (режим энкодера). Во-вторых, названия параметров верхнего уровня: model - для выбора исплоьзуемой модели, messages - промпт, temperature - случайность генерации, tools - спсиок доступных инструментов с описанием, stream - формат вывода  (ответ разом или по одному токену), response_format - формат ответа. В третьих, формат JSON-ответа. Он должен вернуть, причину завершения, статистику по использованным токенам.

```
from openai import OpenAI

client = OpenAI(api_key, base_url) # Инициализация клиента
response = client.chat.completions.create(
    model="llama3",
    messages=[
        {"role": "system", "content": "Ты полезный ассистент."},
        {"role": "user", "content": "Привет!"}
    ],
    temperature=0.7
) # Базовый запрос

print(response.choices[0].message.content) # Извлечение ответа
```

## Бекенды

## Pytorch

### TensorRT и TensorRT-LLM

### Triton Inference Server

### Triton
Язык описания CUDA ядер от Nvidia. Более высокоуровневый чем чистый C++.

### BentoML


## Langchain и LangFraph
afa